<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">

# Python & AI in Asset Management
## Chapter 5 · CAPM, Multifactor Models, and APT

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by GPT 5.1<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook operationalizes Chapter 5 by:
- computing excess returns for the asset universe,
- estimating CAPM betas via time-series regressions,
- plotting rolling betas, and
- extending to a simple multi-factor model using <code>SPY</code>, <code>GLD</code>, and <code>TLT</code> as factors.


### Getting Help While Modeling
- Revisit **Chapter 3** for pandas and plotting refreshers.
- Use **Appendix B** for quick NumPy/pandas syntax help.
- Peek at **Appendix A** if you want the derivations behind CAPM/linear regression formulas.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from pathlib import Path

plt.style.use("seaborn-v0_8")
plt.rcParams.update({"font.family": "serif", "figure.dpi": 300})

DATA_PATH = Path("../data/pyaiam_eod.csv")
if not DATA_PATH.exists():
    DATA_PATH = "https://hilpisch.com/pyaiam_eod.csv"
RISK_FREE_ANNUAL = 0.02
RISK_FREE_DAILY = RISK_FREE_ANNUAL / 252

## 1. Load Prices and Compute Excess Returns
We treat <code>SPY</code> as the benchmark proxy and subtract a constant risk-free rate to obtain excess returns.

In [ ]:
prices = pd.read_csv(DATA_PATH, parse_dates=["Date"]).set_index("Date").sort_index()
prices = prices.ffill()
log_rets = np.log(prices / prices.shift(1)).dropna()
excess_rets = log_rets.subtract(RISK_FREE_DAILY)
excess_rets.head()

### 1.1 Helper for Regression Data
This keeps CAPM regressions tidy by aligning the asset and benchmark series.

In [ ]:
def regression_frame(asset: str, market: str = "SPY") -> pd.DataFrame:
    frame = pd.DataFrame({"asset": excess_rets[asset], "market": excess_rets[market]})
    return frame.dropna()

capm_data = regression_frame("AAPL")
capm_data.head()

## 2. CAPM Regression (AAPL vs. SPY)
We run an OLS regression of excess asset returns on market excess returns and report alpha/beta.

In [ ]:
X = sm.add_constant(capm_data["market"])
y = capm_data["asset"]
model = sm.OLS(y, X).fit()
pd.DataFrame({"param": model.params, "tvalue": model.tvalues})

### 2.1 Rolling Beta Diagnostics
A 1-year rolling window illustrates how beta evolves through time.

In [ ]:
window = 252
cov = log_rets[["AAPL", "SPY"]].rolling(window).cov().dropna()
rolling_beta = (
    cov.xs("AAPL", level=1)["SPY"] /
    cov.xs("SPY", level=1)["SPY"]
)
fig, ax = plt.subplots(figsize=(12, 6))
rolling_beta.plot(ax=ax)
ax.set_title("Rolling 1Y Beta (AAPL vs. SPY)")
ax.set_ylabel("Beta")
plt.show()

## 3. Multifactor Example
We treat <code>SPY</code>, <code>GLD</code>, and <code>TLT</code> as market, commodity, and rate factors, respectively.

In [ ]:
factors = excess_rets[["SPY", "GLD", "TLT"]]
asset = excess_rets["NVDA"]
X = sm.add_constant(factors)
multi_model = sm.OLS(asset, X).fit()
pd.DataFrame({"param": multi_model.params, "tvalue": multi_model.tvalues})

### 3.1 Factor Premia and Expected Returns
We combine estimated loadings with factor means to estimate expected returns for each asset.

In [ ]:
factor_means = factors.mean() * 252
loadings = {}
for ticker in excess_rets.columns:
    X = sm.add_constant(factors)
    res = sm.OLS(excess_rets[ticker], X).fit()
    loadings[ticker] = res.params[1:]
loadings_df = pd.DataFrame(loadings).T
expected_returns = loadings_df @ factor_means
expected_returns.head()

## 4. Exercises
### Exercise 1 – CAPM for Every Asset
Write a function that returns a DataFrame of alpha/beta estimates for all tickers.
<details><summary>Hint</summary>
Iterate through <code>excess_rets.columns</code>, reuse <code>regression_frame</code>, and collect <code>model.params</code> results.
</details>

### Exercise 2 – Beta vs. Volatility Plot
Plot beta on the x-axis and annualized volatility on the y-axis for each asset, labeling points.
<details><summary>Hint</summary>
Use the alpha/beta table from Exercise 1 and combine with <code>log_rets.std()</code>.
</details>

### Exercise 3 – Custom Factor Set
Pick any two assets as new factors (e.g., <code>BTC-USD</code> and <code>EURUSD</code>) and rerun the multi-factor regression for <code>JPM</code>.
<details><summary>Hint</summary>
Construct <code>X = sm.add_constant(excess_rets[[factor1, factor2]])</code> and interpret coefficients.
</details>


## 5. Takeaways for Chapter 5
- pandas + statsmodels make CAPM/MF regressions turnkey.
- Rolling diagnostics reveal instability that static tables hide.
- Simple factor mixes (SPY/GLD/TLT) already show how to assemble factor-based expected returns.


<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">